# 01 Dasymetric population

We disaggregate 2020 census population to MapPLUTO 21v3 tax lots with the cadastral-based expert dasymetric system (CEDS) of Maantay et al. (2007). We split lots at census boundaries, select residential units or adjusted residential area as the proxy for each block group, and validate the surface against block counts. The resulting lot pieces serve as demand points in `02_network_accessibility`.

We depart from the original method in five ways. We use 2020 P1 counts, which carry TopDown Algorithm noise where the 2000 block counts carried suppression. We use MapPLUTO 21v3 in place of LotInfo, and since 21v3 carries only 2010 census fields, we split lots at 2020 census boundaries rather than assigning them by identifier. We reallocate population by lot area in zones with no residential proxy, a case the original does not address. We build the filtered areal weighting mask from PLUTO land use rather than TIGER landmark files, and we add a held-out validation at the block level.

PLUTO aggregates condominium units to the billing lot, so resolution in condominium-heavy areas stops at the billing lot. We remove the Rikers Island tract as the original did, and we flag other group quarters by building class without removing them.

We read the shoreline-clipped MapPLUTO 21v3 from `data/raw/`, since underwater lots in the unclipped release inflate the proxy totals. Block population and block geometry download on the first run and cache to `data/raw/`, and the Census API key reads from the `CENSUS_API_KEY` environment variable.

---
## Setup

In [3]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import requests
from urllib.parse import urlencode, quote

sys.path.append("../src")
from paths import RAW, PROCESSED, CRS_PROJECTED

pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
warnings.filterwarnings("ignore", category=UserWarning, module="geopandas")

In [4]:
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY")

MAPPLUTO = RAW / "MapPLUTO_21v3.gdb"          # shoreline-clipped release
MAPPLUTO_LAYER = "MapPLUTO_21v3_clipped"
BLOCKS_2020 = RAW / "tl_2020_36_tabblock20.zip"
BLOCK_POP_CSV = RAW / "dec2020_p1_blocks_ny.csv"

STATE_FIPS = "36"
NYC_COUNTIES = {"005": "Bronx", "047": "Brooklyn", "061": "Manhattan",
                "081": "Queens", "085": "Staten Island"}
BORO_NAMES = {"MN": "Manhattan", "BX": "Bronx", "BK": "Brooklyn",
              "QN": "Queens", "SI": "Staten Island"}

# disaggregate the coarse level, select the proxy at the finer level
SOURCE_LEVEL = "tract"
VALID_LEVEL = "bg"
LEVEL_COL = {"tract": "tract_geoid", "bg": "bg_geoid", "block": "block_geoid"}

RIKERS_TRACT = "36005000100"

---
## 1. Cadastral data

We read MapPLUTO by layer name from the file geodatabase, which returns full field names rather than the ten-character truncation of a shapefile.

In [6]:
COLS = ["BBL", "ResArea", "UnitsRes", "BldgArea", "UnitsTotal",
        "LotArea", "BldgClass", "LandUse", "Borough", "PLUTOMapID"]

lots = gpd.read_file(MAPPLUTO, layer=MAPPLUTO_LAYER, columns=COLS)
lots = lots.to_crs(CRS_PROJECTED)

for c in ["ResArea", "UnitsRes", "BldgArea", "UnitsTotal", "LotArea", "PLUTOMapID"]:
    lots[c] = pd.to_numeric(lots[c], errors="coerce").fillna(0)

lots["BBL"] = pd.to_numeric(lots["BBL"], errors="coerce").astype("Int64").astype(str)

print(f"{len(lots):,} lots  |  {lots['UnitsRes'].sum():,.0f} residential units")

857,229 lots  |  3,621,078 residential units


---
## 2. Census data

We download population from table P1 of the 2020 PL 94-171 file at the block level, one API call per borough, and take block geometry from TIGER/Line 2020. We aggregate blocks to block groups and tracts so the three levels stay internally consistent. A Census API key is optional at this volume, and both files cache to `data/raw/` after the first run.

In [8]:
if BLOCK_POP_CSV.exists():
    block_pop = pd.read_csv(BLOCK_POP_CSV, dtype={"block_geoid": str})
    print(f"cached: {len(block_pop):,} blocks")
else:
    frames = []
    for fips, name in NYC_COUNTIES.items():
        params = {"get": "P1_001N", "for": "block:*",
                  "in": f"state:{STATE_FIPS} county:{fips} tract:*"}
        if CENSUS_API_KEY:
            params["key"] = CENSUS_API_KEY
        # quote_via=quote gives %20 for spaces; the API rejects +
        url = f"https://api.census.gov/data/2020/dec/pl?{urlencode(params, quote_via=quote, safe=':*')}"

        r = requests.get(url, timeout=180)
        r.raise_for_status()
        rows = r.json()
        df = pd.DataFrame(rows[1:], columns=rows[0])
        df["block_geoid"] = df["state"] + df["county"] + df["tract"] + df["block"]
        df["pop"] = pd.to_numeric(df["P1_001N"], errors="coerce").fillna(0).astype(int)
        frames.append(df[["block_geoid", "pop"]])
        print(f"{name:14} {len(df):>7,} blocks  {df['pop'].sum():>10,} people")

    block_pop = pd.concat(frames, ignore_index=True)
    block_pop.to_csv(BLOCK_POP_CSV, index=False)
    print(f"\n{block_pop['pop'].sum():,} people -> {BLOCK_POP_CSV.name}")

cached: 37,984 blocks


In [9]:
if not BLOCKS_2020.exists():
    url = (f"https://www2.census.gov/geo/tiger/TIGER2020PL/LAYER/TABBLOCK/2020/"
           f"tl_2020_{STATE_FIPS}_tabblock20.zip")
    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(BLOCKS_2020, "wb") as f:
            for part in r.iter_content(chunk_size=1 << 20):
                f.write(part)
    print(f"{BLOCKS_2020.stat().st_size / 1e6:.0f} MB")

In [10]:
blocks = gpd.read_file(BLOCKS_2020)
blocks = blocks[blocks["COUNTYFP20"].isin(NYC_COUNTIES)].to_crs(CRS_PROJECTED)

blocks["block_geoid"] = blocks["GEOID20"].astype(str).str.zfill(15)
blocks["tract_geoid"] = blocks["block_geoid"].str[:11]
blocks["bg_geoid"] = blocks["block_geoid"].str[:12]

block_pop["block_geoid"] = block_pop["block_geoid"].astype(str).str.zfill(15)
blocks = blocks.merge(block_pop, on="block_geoid", how="left")

unmatched = blocks["pop"].isna().sum()
blocks["pop"] = blocks["pop"].fillna(0)
blocks = blocks[["block_geoid", "bg_geoid", "tract_geoid", "pop", "geometry"]]

print(f"{len(blocks):,} blocks  |  {blocks['pop'].sum():,.0f} people  |  {unmatched:,} unmatched")

37,984 blocks  |  8,804,190 people  |  0 unmatched


In [11]:
bg = blocks.groupby("bg_geoid")["pop"].sum().reset_index()
bg["tract_geoid"] = bg["bg_geoid"].str[:11]

tract = blocks.groupby("tract_geoid")["pop"].sum().reset_index()

CENSUS = {"block": blocks, "bg": bg, "tract": tract}

---
## 3. Preparation

We remove underwater lots, which number zero in the clipped release. We flag likely group quarters by building class and carry the flag into the exported surface, since PLUTO records no residential units for most group quarters while the census counts their residents. The flag does not change how we allocate population.

In [13]:
# building classes with residents the cadastral proxies miss
GQ_CLASSES = {
    "H8": "dormitory", "HR": "SRO", "HH": "hostel",
    "I1": "hospital/sanitarium", "I6": "nursing home", "I7": "adult care facility",
    "N1": "asylum", "N2": "home for indigent/aged/homeless", "N3": "orphanage",
    "N4": "detention house",
    "W5": "city university", "W6": "other college/university", "W7": "theological seminary",
    "Y3": "prison/jail/house of detention", "Y4": "military installation",
}

n_before = len(lots)
lots = lots[~lots["PLUTOMapID"].isin([4, 5])].copy()          # underwater
lots["gq_flag"] = lots["BldgClass"].str[:2].map(GQ_CLASSES)

print(f"{n_before - len(lots):,} underwater lots dropped")
print(f"{lots['gq_flag'].notna().sum():,} group quarters flagged")

0 underwater lots dropped
1,364 group quarters flagged


### Eq. 1: adjusted residential area

$$ARA = M \cdot \left(BA \cdot \frac{RU}{TU}\right) + RA, \qquad
M = \begin{cases} 1 & \text{if } RA = 0 \text{ and } RU \neq 0 \\ 0 & \text{otherwise}\end{cases}$$

We impute residential floor area where a lot records residential units but no residential area, apportioning building area by the residential share of units. Where `TU = 0` the formula divides by zero, so we assign the full building area, since a lot with residential units and no recorded total units is almost always wholly residential; the fallback applies to no lots in 21v3. Imputation cannot help the 436 lots that record residential units but no building area, most of them class G0, so those lots receive population only where residential units win as the proxy.

In [15]:
ra = lots["ResArea"].astype(float)
ru = lots["UnitsRes"].astype(float)
ba = lots["BldgArea"].astype(float)
tu = lots["UnitsTotal"].astype(float)

m = (ra == 0) & (ru != 0)
ratio = np.where(tu > 0, ru / tu.replace(0, np.nan), 1.0)   # TU=0 -> treat as wholly residential
ratio = np.clip(ratio, 0, 1)                                # RU > TU occurs as a data error

lots["ARA"] = np.where(m, ba * ratio, 0.0) + ra
lots["RU"] = ru

print(f"lots imputed:        {m.sum():,} ({m.mean():.2%})")
print(f"TU=0 fallback:       {(m & (tu == 0)).sum():,}")
print(f"RU > TU:             {(ru > tu).sum():,}")
print(f"ARA=0 but RU>0:      {((lots['ARA'] == 0) & (ru > 0)).sum():,}")

lots imputed:        5,358 (0.63%)
TU=0 fallback:       0
RU > TU:             0
ARA=0 but RU>0:      436


In [16]:
# lots with units but no area after Eq. 1
z = lots[(lots["ARA"] == 0) & (lots["RU"] > 0)]
print(f"{z['RU'].sum():,.0f} residential units on {len(z)} lots  |  building area {z['BldgArea'].sum():,.0f} sq ft")
print(z["Borough"].value_counts().to_string())
print(z["BldgClass"].value_counts().head(5).to_string())

996 residential units on 436 lots  |  building area 0 sq ft
Borough
QN    193
BK    172
SI     61
BX      7
MN      3
BldgClass
G0    422
Z0      2
D6      2
Z9      2
N2      2


---
## 4. Assignment to 2020 census geography

MapPLUTO 21v3 carries only 2010 census identifiers, so we assign lots to 2020 geography spatially. We split each lot at census boundaries and apportion residential units, adjusted residential area, and lot area to each piece by its share of the lot's area, so a lot crossing a boundary contributes to both zones. We treat pieces as the unit of analysis from here on, and we remove the Rikers Island tract, where the census counts an incarcerated population on lots PLUTO does not code as residential.

In [18]:
lots["lot_id"] = np.arange(len(lots))
lots["lot_area"] = lots.geometry.area

pieces = gpd.overlay(
    lots[["lot_id", "lot_area", "geometry"]],
    blocks[["block_geoid", "bg_geoid", "tract_geoid", "geometry"]],
    how="intersection", keep_geom_type=True,
)
pieces["piece_area"] = pieces.geometry.area
pieces["frac"] = (pieces["piece_area"] / pieces["lot_area"].replace(0, np.nan)).fillna(0)

pieces = pieces.merge(lots.drop(columns=["geometry", "lot_area"]), on="lot_id", how="left")

for c in ["RU", "ARA", "LotArea"]:
    pieces[c] = pieces[c] * pieces["frac"]

print(f"{len(lots):,} lots -> {len(pieces):,} pieces")
print(f"{(pieces.groupby('lot_id').size() > 1).sum():,} lots split")
print(f"area retained:  {pieces['piece_area'].sum() / lots['lot_area'].sum():.4f}")
print(f"units retained: {pieces['RU'].sum() / lots['RU'].sum():.4f}")

857,229 lots -> 884,157 pieces
22,104 lots split
area retained:  0.9996
units retained: 1.0000


In [19]:
n_before = len(pieces)
pieces = pieces[pieces["tract_geoid"] != RIKERS_TRACT]
n_rikers = n_before - len(pieces)

pieces = pieces[pieces["block_geoid"].notna()].copy()

print(f"{n_rikers:,} Rikers pieces dropped")
print(f"{n_before - n_rikers - len(pieces):,} unassigned pieces dropped")

7 Rikers pieces dropped
0 unassigned pieces dropped


---
## 5. Eq. 2: dasymetric calculation

$$POP_l = POP_c \cdot \frac{U_l}{U_c}$$

$U_l$ is the proxy on piece $l$ and $U_c$ is the proxy total in the census zone containing it.

Maantay et al. (2007) do not address zones with population but no proxy on any lot. In 2020 NYC we find them mostly on institutional lots, such as jails, colleges, and hospitals, where the census counts residents that PLUTO does not record as residential units, and where differential privacy places a few people in an empty block. We reallocate their population by lot area through `FALLBACK`, since removing them would violate the pycnophylactic property and delete population from the neighborhoods the equity analysis concerns.

In [21]:
FALLBACK = "lotarea"          # lotarea, bldgarea, equal, or drop

def disaggregate(df, zone_col, proxy_col, zone_pop):
    proxy = df[proxy_col].astype(float).clip(lower=0)
    zone_total = proxy.groupby(df[zone_col]).transform("sum")
    zpop = df[zone_col].map(zone_pop).astype(float).fillna(0.0)

    pop = zpop * np.where(zone_total > 0, proxy / zone_total.replace(0, np.nan), np.nan)

    # Zones with population but no proxy anywhere: reallocate rather than drop
    dead = (zone_total <= 0) & (zpop > 0)
    if dead.any():
        alt = {"lotarea": df["LotArea"].astype(float).clip(lower=0),
               "bldgarea": df["BldgArea"].astype(float).clip(lower=0),
               "equal": pd.Series(1.0, index=df.index),
               "drop": None}[FALLBACK]

        if alt is None:
            pop = np.where(dead, 0.0, pop)
        else:
            alt_total = alt.groupby(df[zone_col]).transform("sum")
            alt_share = np.where(alt_total > 0, alt / alt_total.replace(0, np.nan), 0.0)
            pop = np.where(dead, zpop * alt_share, pop)

    diag = {"zones": int(df.loc[dead, zone_col].nunique()),
            "pieces": int(dead.sum()),
            "population": float(zpop[dead].groupby(df.loc[dead, zone_col]).first().sum())}

    return pd.Series(np.nan_to_num(pop, nan=0.0), index=df.index), diag

---
## 6. Eqs. 3 and 4: expert system

We select the proxy for each block group rather than for the whole city. We disaggregate tract population by residential units and again by adjusted residential area (A), aggregate each estimate to the block group and compare it with the census count (B), and keep the proxy whose estimate falls closer (C). We then disaggregate block group population with each block group's chosen proxy (D), which gives the final surface.

$$POP_{diff} = |POP_{valid} - POP_{est}| \qquad \text{(Eq. 3)}$$

$$\text{IF } RU\_POP_{diff} \leq ARA\_POP_{diff} \text{ THEN } POP_l = POP_{RU} \text{ ELSE } POP_l = POP_{ARA} \qquad \text{(Eq. 4)}$$

In [23]:
src_col, val_col = LEVEL_COL[SOURCE_LEVEL], LEVEL_COL[VALID_LEVEL]
src_pop = CENSUS[SOURCE_LEVEL].set_index(src_col)["pop"]
val_pop = CENSUS[VALID_LEVEL].set_index(val_col)["pop"]

ceds_diags = {}

# A: disaggregate the coarse level with each proxy
for proxy in ["RU", "ARA"]:
    pieces[f"pop_src_{proxy}"], ceds_diags[f"src_{proxy}"] = disaggregate(
        pieces, src_col, proxy, src_pop)

# B: re-aggregate to the validation level (Eq. 3)
selection = pieces.groupby(val_col)[["pop_src_RU", "pop_src_ARA"]].sum()
selection.columns = ["est_RU", "est_ARA"]
selection["census_pop"] = val_pop.reindex(selection.index).fillna(0)
selection["RU_POPdiff"] = (selection["census_pop"] - selection["est_RU"]).abs()
selection["ARA_POPdiff"] = (selection["census_pop"] - selection["est_ARA"]).abs()

# C: per-zone winner (Eq. 4)
selection["chosen_proxy"] = np.where(
    selection["RU_POPdiff"] <= selection["ARA_POPdiff"], "RU", "ARA")

# D: final disaggregation, from the validation level
for proxy in ["RU", "ARA"]:
    pieces[f"pop_val_{proxy}"], ceds_diags[f"val_{proxy}"] = disaggregate(
        pieces, val_col, proxy, val_pop)

pieces["chosen_proxy"] = pieces[val_col].map(selection["chosen_proxy"])
pieces["pop_ceds"] = np.where(pieces["chosen_proxy"].eq("RU"),
                              pieces["pop_val_RU"], pieces["pop_val_ARA"])

print(selection["chosen_proxy"].value_counts(normalize=True).to_string(float_format="{:.1%}".format))
print(f"\nCEDS:   {pieces['pop_ceds'].sum():,.0f}")
print(f"census: {val_pop.sum():,.0f}")

chosen_proxy
RU    57.7%
ARA   42.3%

CEDS:   8,800,353
census: 8,804,190


In [24]:
# block groups where the chosen proxy is zero, so FALLBACK allocated the population
chosen = pd.Series(np.where(pieces["chosen_proxy"].eq("RU"), pieces["RU"], pieces["ARA"]),
                   index=pieces.index)
chosen_total = chosen.groupby(pieces[val_col]).sum()
dead = chosen_total[chosen_total <= 0].index.intersection(val_pop[val_pop > 0].index)

print(f"reallocated by {FALLBACK}: {len(dead)} block groups, {val_pop[dead].sum():,.0f} people")
print(pd.DataFrame(ceds_diags).T.to_string())

d = CENSUS["bg"][CENSUS["bg"]["bg_geoid"].isin(dead)].copy()
d["boro"] = d["bg_geoid"].str[2:5].map(NYC_COUNTIES)
print(d.groupby("boro")["pop"].agg(["size", "sum"]).to_string())

reallocated by lotarea: 83 block groups, 10,492 people
         zones    pieces  population
src_RU  36.000 1,601.000   1,929.000
src_ARA 32.000 1,408.000   1,780.000
val_RU  86.000 2,166.000  10,577.000
val_ARA 79.000 1,960.000  10,343.000
               size   sum
boro                     
Bronx             9   136
Brooklyn         20  4204
Manhattan        26  5019
Queens           25  1077
Staten Island     3    56


In [25]:
# land use on the largest fallback block groups
for bgid in d.nlargest(4, "pop")["bg_geoid"]:
    sub = pieces[pieces["bg_geoid"] == bgid]
    print(f"\n{bgid}  ({val_pop[bgid]:,.0f} people, {len(sub)} pieces)")
    print(sub[["BldgClass", "LandUse"]].value_counts().head(5).to_string())


360470018012  (2,562 people, 1 pieces)
BldgClass  LandUse
Y3         08         1

360610203002  (1,981 people, 2 pieces)
BldgClass  LandUse
W6         08         2

360610240002  (1,300 people, 11 pieces)
BldgClass  LandUse
Q1         09         10
Y1         08          1

360610062001  (953 people, 17 pieces)
BldgClass  LandUse
I1         08         4
U6         07         2
Q1         09         2
I9         08         1
P6         09         1


---
## 7. Control: filtered areal weighting

$$POP_{FAW} = POP_{TR} \cdot \frac{AREA_{BG}}{AREA_{TR}}$$

We benchmark CEDS against filtered areal weighting, which masks uninhabited land and splits tract population among block groups by the remaining area. Maantay et al. (2007) built the mask from TIGER landmark and water files, and we build it from PLUTO land use, which is finer and gives a stricter comparison. `MASK = "none"` removes the mask and approximates the cruder original control.

In [27]:
# transport/utility, open space, parking, vacant
UNINHABITED_LANDUSE = {"07", "09", "10", "11"}
MASK = "pluto"          # "none" disables it

habitable = (pd.Series(True, index=pieces.index) if MASK == "none"
             else ~pieces["LandUse"].astype(str).str.zfill(2).isin(UNINHABITED_LANDUSE))

hab_area = np.where(habitable, pieces["LotArea"].astype(float), 0.0)

area_val = pd.Series(hab_area, index=pieces.index).groupby(pieces[val_col]).sum()
area_src = pd.Series(hab_area, index=pieces.index).groupby(pieces[src_col]).sum()
val_to_src = pieces.groupby(val_col)[src_col].first()

faw = (val_to_src.map(src_pop) *
       (area_val / val_to_src.map(area_src).replace(0, np.nan))).fillna(0.0)
faw.name = "est_FAW"

print(f"FAW total: {faw.sum():,.0f}")

FAW total: 8,800,198


---
## 8. Validation

We report the percent absolute difference, $\sum |est - census| / \sum census \times 100$, for CEDS, each proxy alone, and filtered areal weighting against block group counts. Maantay et al. (2007) report 6.37 for CEDS and 21.91 for filtered areal weighting on 2000 data, alongside a regression through the origin. SPSS reports an uncentered $R^2$ for regressions without an intercept, which runs higher than the centered $R^2$, so we compute both and compare the uncentered value with the original.

In [29]:
def regress_through_origin(est, obs):
    x, y = np.asarray(obs, float), np.asarray(est, float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]

    b = (x * y).sum() / (x * x).sum()
    ss_res = ((y - b * x) ** 2).sum()

    return {"slope": b,
            "r2_uncentered": 1 - ss_res / (y ** 2).sum(),        # SPSS-style
            "r2_centered": 1 - ss_res / ((y - y.mean()) ** 2).sum(),
            "std_error": np.sqrt(ss_res / (len(x) - 1)),
            "n": len(x)}


def pct_abs_diff(est, obs):
    est, obs = np.asarray(est, float), np.asarray(obs, float)
    return 100.0 * np.abs(est - obs).sum() / obs.sum()

In [30]:
obs = selection["census_pop"]
methods = {
    "Filtered Areal Weighting": faw.reindex(selection.index).fillna(0),
    "Adjusted Residential Area": selection["est_ARA"],
    "Residential Units": selection["est_RU"],
    "CEDS": np.where(selection["chosen_proxy"].eq("RU"),
                     selection["est_RU"], selection["est_ARA"]),
}

rows = []
for name, est in methods.items():
    r = regress_through_origin(est, obs)
    r["method"] = name
    r["pct_abs_diff"] = pct_abs_diff(est, obs)
    rows.append(r)

results = pd.DataFrame(rows).set_index("method")[
    ["pct_abs_diff", "slope", "r2_uncentered", "r2_centered", "std_error", "n"]]
results

,pct_abs_diff,slope,r2_uncentered,r2_centered,std_error,n
method,,,,,,
Filtered Areal Weighting,20.451,0.971,0.920,0.619,414.399,6771
Adjusted Residential Area,10.864,0.993,0.971,0.857,248.099,6771
Residential Units,10.020,0.994,0.974,0.873,233.111,6771
CEDS,7.345,0.994,0.984,0.919,181.617,6771


### Held-out validation

Eq. 4 selects each block group's proxy from the same census counts we evaluate against, so the block group comparison favors CEDS by construction. We validate a second time at the block level, a finer geography the selection rule does not observe, comparing CEDS with each proxy alone, all three disaggregated from block group population.

In [32]:
block_pop_s = CENSUS["block"].set_index("block_geoid")["pop"]
bg_pop = CENSUS["bg"].set_index("bg_geoid")["pop"]

est_blk = {"CEDS": pieces.groupby("block_geoid")["pop_ceds"].sum()}
for proxy in ["RU", "ARA"]:
    p, _ = disaggregate(pieces, "bg_geoid", proxy, bg_pop)
    est_blk[proxy] = p.groupby(pieces["block_geoid"]).sum()

blk = pd.DataFrame(est_blk)
obs_blk = block_pop_s.reindex(blk.index).fillna(0)

rows = []
for col, name in [("CEDS", "CEDS"), ("RU", "Residential Units"), ("ARA", "Adjusted Residential Area")]:
    r = regress_through_origin(blk[col], obs_blk)
    r["method"] = name
    r["pct_abs_diff"] = pct_abs_diff(blk[col], obs_blk)
    rows.append(r)

holdout = pd.DataFrame(rows).set_index("method")[
    ["pct_abs_diff", "slope", "r2_uncentered", "std_error", "n"]]
holdout

,pct_abs_diff,slope,r2_uncentered,std_error,n
method,,,,,
CEDS,13.275,0.992,0.966,74.810,35224
Residential Units,13.542,0.992,0.963,78.932,35224
Adjusted Residential Area,13.784,0.989,0.965,75.911,35224


---
## 9. Diagnostics

We check that population is conserved within every block group, account for the citywide shortfall against the census total, and check the implied persons per residential unit where residential units allocated population.

In [34]:
summed = pieces.groupby(val_col)["pop_ceds"].sum()
resid = (summed - val_pop.reindex(summed.index).fillna(0)).abs()
no_pieces = val_pop.index.difference(summed.index)

print(f"zones off by more than 1: {(resid > 1).sum():,} of {len(resid):,}")
print(f"CEDS:   {pieces['pop_ceds'].sum():,.0f}")
print(f"census: {val_pop.sum():,.0f}")
print(f"block groups with no pieces: {len(no_pieces)}, {val_pop[no_pieces].sum():,.0f} people")
print(val_pop[no_pieces][val_pop[no_pieces] > 0].to_string())

zones off by more than 1: 0 of 6,771
CEDS:   8,800,353
census: 8,804,190
block groups with no pieces: 36, 3,837 people
bg_geoid
360050001001    3772
360610167000      34
360610191000      31


In [35]:
# RU-selected zones only; pop/RU means nothing where ARA allocated
ru_zones = selection.index[selection["chosen_proxy"] == "RU"]
sub = pieces[pieces[val_col].isin(ru_zones)]

parcels = sub.groupby("BBL").agg(pop=("pop_ceds", "sum"), ru=("RU", "sum"))
parcels = parcels[parcels["ru"] >= 1]
pph = parcels["pop"] / parcels["ru"]

print(f"RU-selected zones: {len(ru_zones):,} of {len(selection):,}")
print(f"median {pph.median():.2f}  |  p05 {pph.quantile(.05):.2f}  |  p95 {pph.quantile(.95):.2f}")
print(f"parcels above 10 persons/unit: {(pph > 10).sum():,}")
print(f"citywide implied, all zones: {pieces['pop_ceds'].sum() / pieces['RU'].sum():.2f}")

RU-selected zones: 3,910 of 6,771
median 2.90  |  p05 1.83  |  p95 3.92
parcels above 10 persons/unit: 26
citywide implied, all zones: 2.43


---
## 10. Export

We write the lot piece surface as a parquet of representative points, which `02_network_accessibility` reads and filters to demand points, and as a GeoPackage of piece polygons for mapping.

In [37]:
COLS_OUT = ["BBL", "Borough", "block_geoid", "bg_geoid", "tract_geoid",
            "RU", "ARA", "LotArea", "BldgClass", "LandUse",
            "gq_flag", "chosen_proxy", "pop_ceds", "geometry"]

surface = pieces[COLS_OUT].copy()
surface["piece_id"] = np.arange(len(surface))          # BBL repeats across pieces

surface.to_file(PROCESSED / "ceds_pop_nyc_2020.gpkg", layer="ceds_pieces", driver="GPKG")

flat = surface.drop(columns="geometry")
cent = surface.geometry.representative_point()
flat["x"], flat["y"] = cent.x.values, cent.y.values
flat.to_parquet(PROCESSED / "ceds_pop_nyc_2020.parquet", index=False)

print(f"{len(surface):,} pieces  |  {surface['pop_ceds'].sum():,.0f} people")

884,150 pieces  |  8,800,353 people
